In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.inference import Predictor, sweep_fid_vs_anchor_count

In [2]:
RUN_DIR = "run/20260422_143719"  # set directly — adjust to your run
VOLUME_PATH = "data/default/3d/sample_volume.npy"
K_LIST = None  # None → 0..train_shape[0] inclusive (resolved after predictor loads)
N_PER_K = 4    # for reliable FID raise above ceil(2048 / (D+H+W))
SEED_BASE = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
run_dir = str(REPO_ROOT / RUN_DIR)
predictor = Predictor(run_dir=run_dir, device=DEVICE)
gt_volume = np.load(REPO_ROOT / VOLUME_PATH)

expected_grayscale = predictor.in_channels == 1 and gt_volume.ndim == 3
expected_rgb = (
    predictor.in_channels == 3
    and gt_volume.ndim == 4
    and gt_volume.shape[-1] == 3
)
assert expected_grayscale or expected_rgb, (
    f"GT shape {gt_volume.shape} incompatible with in_channels={predictor.in_channels}"
)
assert gt_volume.dtype == np.uint8, f"GT must be uint8; got {gt_volume.dtype}"
assert gt_volume.shape[:3] == tuple(predictor.train_shape), (
    f"GT shape {gt_volume.shape[:3]} != train_shape {tuple(predictor.train_shape)}"
)

if K_LIST is None:
    K_LIST = list(range(0, predictor.train_shape[0] + 1))

print("run_dir     :", run_dir)
print("device      :", DEVICE)
print("train_shape :", predictor.train_shape)
print("in_channels :", predictor.in_channels)
print("GT shape    :", gt_volume.shape)
print("GT dtype    :", gt_volume.dtype)
print("GT range    :", (int(gt_volume.min()), int(gt_volume.max())))
print("K range     :", f"{K_LIST[0]}..{K_LIST[-1]} ({len(K_LIST)} values)")

FileNotFoundError: [Errno 2] No such file or directory: '/home/kn/code/conditional-slice-gan/run/20260422_143719/config.yaml'

In [ ]:
results = sweep_fid_vs_anchor_count(
    predictor=predictor,
    gt_volume=gt_volume,
    k_list=K_LIST,
    n_per_k=N_PER_K,
    seed_base=SEED_BASE,
    device=DEVICE,
    progress=True,
)
print(f"swept {len(results)} K values, N={N_PER_K} per K")

In [ ]:
ks = [k for k, _ in results]
fids = [f for _, f in results]

print("  K   FID")
print("----  -------")
for k, f in results:
    print(f"{k:4d}  {f:7.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, fids, marker="o")
ax.set_xlabel("anchor count K")
ax.set_ylabel("FID")
ax.set_title(f"FID vs anchor count (N={N_PER_K} per K)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()